This is notebook-ready prototype for the "Smart Home Safety Sentry." This script demonstrates the **"Hybrid Reality"**: **TensorFlow** handles the raw, high-speed vision identification, while **Gemini (via Google ADK)** handles the high-level human reasoning and decision-making.

# Step 1: Install Dependencies
First, we need to install the libraries. In a Jupyter cell, run:

In [35]:
%pip install -q -r ../requirements.txt

import ssl
import os

# Bypass SSL for the current session to download the weights (macOS workaround)
ssl._create_default_https_context = ssl._create_unverified_context
print("Dependencies installed and SSL context configured.")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dependencies installed and SSL context configured.


# Step 2: "The Eyes" - Setting up MobileNetV2
We will create a function that uses a pre-trained MobileNetV2 model. This function is our "Eye"—it doesn't understand "safety," it only understands "pixels to labels."

In [36]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the pre-trained model (trained on ImageNet)
vision_model = MobileNetV2(weights='imagenet')

def identify_object(img_path: str) -> dict:
    """
    Analyzes an image and returns the most likely object detected.
    """
    # ADDED PRINT FOR VISIBILITY
    print(f"  [AI-EYES]: Processing image: {img_path}...")
    
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        preds = vision_model.predict(x, verbose=0)
        _, label, conf = decode_predictions(preds, top=1)[0][0]
        
        detected = label.replace('_', ' ')
        print(f"  [AI-EYES]: Detected '{detected}' with {conf*100:.1f}% confidence.")
        
        return {
            "status": "success",
            "detected_object": detected,
            "confidence": float(conf)
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

# TEST THE EYES: 
print(identify_object("../images/porch_dog.jpg"))
print(identify_object("../images/porch_cat.jpg"))
print(identify_object("../images/porch_bear.jpg"))

  [AI-EYES]: Processing image: ../images/porch_dog.jpg...
  [AI-EYES]: Detected 'German shepherd' with 21.8% confidence.
{'status': 'success', 'detected_object': 'German shepherd', 'confidence': 0.21805629134178162}
  [AI-EYES]: Processing image: ../images/porch_cat.jpg...
  [AI-EYES]: Detected 'tiger cat' with 34.9% confidence.
{'status': 'success', 'detected_object': 'tiger cat', 'confidence': 0.34935542941093445}
  [AI-EYES]: Processing image: ../images/porch_bear.jpg...
  [AI-EYES]: Detected 'brown bear' with 86.9% confidence.
{'status': 'success', 'detected_object': 'brown bear', 'confidence': 0.8691624999046326}


# Step 3: "The Brain" - Setting up the Gemini Agent (ADK)
Now we define the Agent. We tell Gemini who it is and give it the identify_object function as a Tool. Gemini will "call" this tool when it needs to "see."

In [40]:
import os
from dotenv import load_dotenv

# Load key from .env file
load_dotenv(dotenv_path="../.env")

import asyncio
from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# --- CONFIGURATION ---
APP_NAME = "SmartSentry_v1"
USER_ID = "demo_user"
SESSION_ID = "session_001"

# --- 1. THE BRAIN & ENGINE ---
sentry_agent = Agent(
    name="HomeSentry",
    model="gemini-2.0-flash", 
    instruction="""
    You are a Smart Home Security Assistant. Use the 'identify_object' tool to see.
    - 'backpack' = Package.
    - 'dog'/'cat' = Pet.
    - 'person' = Visitor.
    Always use your tool before answering. Describe the detected object for the user to have a better understanding of any risk or danger or can it be ignored.
    """,
    tools=[identify_object] # Your TensorFlow function
)

session_service = InMemorySessionService()

runner = Runner(
    app_name=APP_NAME,
    agent=sentry_agent,
    session_service=session_service
)


# Step 4: The "Hybrid" Moment - Run the Prototype
This is the final cell where the two systems shake hands. We ask a natural language question, and the Agent decides to use the TensorFlow tool to answer.

In [41]:
from google.genai import types

async def run_demo(user_query, image_file):
    print(f"USER: {user_query}")
    print("-" * 30)
    
    try:
        await session_service.get_session(app_name=APP_NAME, session_id=SESSION_ID)
    except Exception:
        await session_service.create_session(
            app_name=APP_NAME, 
            user_id=USER_ID, 
            session_id=SESSION_ID
        )

    content = types.Content(
        role="user",
        parts=[types.Part(text=f"{user_query}. Target file: {image_file}")]
    )
    
    try:
        event_stream = runner.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=content
        )
        
        async for event in event_stream:
            # 1. SHOW THE BRAIN'S DECISION (Tool Call)
            # We check if the event contains a tool call request
            if hasattr(event, 'content') and event.content and event.content.parts:
                for part in event.content.parts:
                    if hasattr(part, 'tool_call') and part.tool_call:
                        print(f"\n[BRAIN]: I need to check the camera. Calling TensorFlow...")

            # 2. SHOW THE FINAL RESPONSE
            if event.is_final_response():
                print(f"\nSENTRY: {event.content.parts[0].text}")
                
    except Exception as e:
        print(f"Execution Error: {e}")

# --- 3. RUN ---
#await run_demo("Is there anything at the door?", "../images/porch_dog.jpg")
#await run_demo("Is there anything at the door?", "../images/porch_cat.jpg")
await run_demo("Is there anything at the door?", "../images/porch_bear.jpg")

USER: Is there anything at the door?
------------------------------
  [AI-EYES]: Processing image: ../images/porch_bear.jpg...
  [AI-EYES]: Detected 'brown bear' with 86.9% confidence.

SENTRY: There is a brown bear at the door. I recommend contacting animal control.

